# matvec — ex1: linear layer forward as matvec

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `matvec`. Running the final beacon cell reports progress against the `PyTorch: matrix-vector product` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: matrix-vector product` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`matvec`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "matvec"
DD_SUBTOPIC = "PyTorch: matrix-vector product"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## matrix-vector product — quick refresher

`(M, N) @ (N,)` returns a **vector** of shape `(M,)`. This is the honest matrix-vector product — no batching, no broadcasting, just linear algebra.

**Vector vs column-matrix.** `(M, N) @ (N, 1)` returns a **matrix** of shape `(M, 1)`. Two extra characters, totally different output rank. A `(M, 1)` matrix is a 2-D tensor — you need an extra `.squeeze(-1)` (or `[:, 0]`) to get back to a `(M,)` vector.

Use the `(M,)` form when the result feeds a 1-D operation (softmax over classes, a 1-D loss); use the `(M, 1)` form when you want a column-vector for concatenation with other matrices.

### Exercise 1 — linear layer forward as matvec

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply `W @ x` where `W: (M, N)` and `x: (N,)` to compute a single linear-layer forward as a true `(M,)` vector (not a `(M, 1)` column matrix).
> Keywords: matvec, linear-layer, shape-discipline, matmul
> ```

**KCs targeted:** `matvec-output-is-1d`, `vector-vs-column-distinction`

Implement `ex1_linear_forward(W, x, b)`. A single-sample linear layer forward:

1. `W` has shape `(M, N)` — the weight matrix.
2. `x` has shape `(N,)` — the input feature vector.
3. `b` has shape `(M,)` — the bias vector.
4. Return `W @ x + b` — must be shape `(M,)`, NOT `(M, 1)`.

Do NOT use `x.unsqueeze(-1)` to make it a column. Pass the 1-D `x` directly to `@` — PyTorch returns a 1-D result.

Input: `W: (M, N)`, `x: (N,)`, `b: (M,)`.
Output: `(M,)` float tensor.

In [ ]:
def ex1_linear_forward(W: Tensor, x: Tensor, b: Tensor) -> Tensor:
    """Compute W @ x + b for vector x; output must be a 1-D vector."""
    raise NotImplementedError()


def _test_ex1():
    # Hand-built: W maps 3 inputs to 2 outputs.
    W = t.tensor([
        [1.0, 2.0, 3.0],
        [4.0, 5.0, 6.0],
    ])  # (2, 3)
    x = t.tensor([1.0, 1.0, 1.0])  # (3,)
    b = t.tensor([10.0, 20.0])     # (2,)
    # W @ x = [1+2+3, 4+5+6] = [6, 15]; + b = [16, 35]
    out = ex1_linear_forward(W, x, b)
    assert out.ndim == 1, f'output must be 1-D vector, got ndim={out.ndim} shape={tuple(out.shape)}'
    assert out.shape == (2,), f'expected (2,), got {tuple(out.shape)}'
    assert t.allclose(out, t.tensor([16.0, 35.0])), f'got {out}'

    # Larger realistic shape.
    M, N = 64, 128
    rng = t.Generator().manual_seed(17)
    Wb = t.randn(M, N, generator=rng)
    xb = t.randn(N, generator=rng)
    bb = t.randn(M, generator=rng)
    out_b = ex1_linear_forward(Wb, xb, bb)
    assert out_b.ndim == 1, f'shape contract violated: ndim={out_b.ndim}'
    assert out_b.shape == (M,)
    # Reference via F.linear (which expects (M,N) weights and produces (M,) for 1-D input).
    import torch.nn.functional as F_
    ref = F_.linear(xb, Wb, bb)
    assert t.allclose(out_b, ref, atol=1e-5), 'must match F.linear'

    # CRITICAL distinction — column-matrix form has a different shape.
    x_col = xb.unsqueeze(-1)  # (N, 1)
    col_result = Wb @ x_col   # (M, 1) — this is the WRONG output rank
    assert col_result.ndim == 2 and col_result.shape == (M, 1), 'sanity: column form returns matrix'
    assert col_result.shape != out_b.shape, 'matvec output must NOT be a column matrix'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_linear_forward(W: Tensor, x: Tensor, b: Tensor) -> Tensor:
    return W @ x + b
```

**The 1-D rule.** When the right-hand operand of `@` is 1-D, PyTorch treats it as a vector and returns a 1-D result. When you `unsqueeze(-1)` to make it `(N, 1)`, you've promoted it to a column matrix and the result is `(M, 1)` — same numbers, different rank.

**Why rank matters downstream.** Many ops (softmax, cross-entropy, BatchNorm in 1-D mode) expect a specific rank. Carrying around `(M, 1)` when you meant `(M,)` causes silent broadcasting bugs that surface much later. Get the rank right at the matvec step.

**Batched version.** For `(B, M, N) @ (B, N) -> (B, M)`, use `torch.einsum('bij,bj->bi', W, x)` or `(W @ x.unsqueeze(-1)).squeeze(-1)`. Plain `@` also works thanks to batched matmul rules, but einsum makes the contract explicit.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()